In [ ]:
# Example 1: Top-level package attributes + version metadata.
import diffusers

version = diffusers.__version__
print(f"diffusers version: {version}")

# diffusers exposes __all__ describing its public API surface
public_api = getattr(diffusers, '__all__', None)
if public_api is not None:
    print(f"diffusers.__all__ contains {len(public_api)} entries")
    print(f"sample: {sorted(public_api)[:8]}")

assert isinstance(version, str) and version, "diffusers.__version__ must be a non-empty string"
assert hasattr(diffusers, 'ConfigMixin'), "diffusers must expose ConfigMixin"


In [ ]:
# Example 2: ConfigMixin — framework-agnostic configuration system.
from diffusers import ConfigMixin
from diffusers.configuration_utils import register_to_config

class TinyConfig(ConfigMixin):
    config_name = "tiny_config.json"

    @register_to_config
    def __init__(self, hidden_size: int = 32, num_layers: int = 2, name: str = "tiny"):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.name = name

cfg = TinyConfig(hidden_size=64, num_layers=4, name="demo")
print(f"config dict: {dict(cfg.config)}")

assert cfg.config.hidden_size == 64
assert cfg.config.num_layers == 4
assert cfg.config.name == "demo"


In [ ]:
# Example 3: Inspecting the scheduler registry without importing torch-only schedulers.
import diffusers

# Find scheduler classes among the public API entries that import successfully.
candidates = [name for name in dir(diffusers) if name.endswith('Scheduler')]
print(f"Found {len(candidates)} *Scheduler-named attributes on diffusers")
print(f"sample: {candidates[:6]}")

# At least a few schedulers should be discoverable (some may be lazy stubs without torch).
assert len(candidates) > 0, "diffusers should expose at least one *Scheduler class name"

# The configuration_utils module loads without optional deps.
from diffusers import configuration_utils
assert hasattr(configuration_utils, 'ConfigMixin')
assert hasattr(configuration_utils, 'register_to_config')


In [ ]:
# Example 4: utils submodule — constants, logging, availability helpers.
from diffusers import utils as diff_utils
from diffusers.utils import logging as diff_logging

logger = diff_logging.get_logger("diffusers-smoke-test")
logger.info("diffusers logger works")

# Boolean availability flags should always be defined (True or False).
flags = {
    name: getattr(diff_utils, name)
    for name in dir(diff_utils)
    if name.startswith('is_') and name.endswith('_available') and callable(getattr(diff_utils, name))
}
print(f"availability helpers exposed: {len(flags)}")
print(f"sample names: {sorted(flags)[:6]}")

assert len(flags) > 0, "diffusers.utils must expose at least one is_*_available helper"
# Each helper returns a bool when invoked.
sample_name = sorted(flags)[0]
assert isinstance(flags[sample_name](), bool)


In [ ]:
# Example 5: ConfigMixin round-trip serialization through a temp directory.
import json
import tempfile
from pathlib import Path

from diffusers import ConfigMixin
from diffusers.configuration_utils import register_to_config

class RoundtripConfig(ConfigMixin):
    config_name = "roundtrip_config.json"

    @register_to_config
    def __init__(self, alpha: float = 0.5, label: str = "orig"):
        self.alpha = alpha
        self.label = label

with tempfile.TemporaryDirectory() as tmp:
    cfg = RoundtripConfig(alpha=0.875, label="saved")
    cfg.save_config(tmp)
    on_disk = Path(tmp) / RoundtripConfig.config_name
    raw = json.loads(on_disk.read_text())
    print(f"persisted keys: {sorted(raw)}")
    loaded = RoundtripConfig.load_config(tmp)

assert raw['alpha'] == 0.875
assert raw['label'] == 'saved'
assert loaded['alpha'] == 0.875
assert loaded['label'] == 'saved'
print("diffusers smoke test passed")
